In [ ]:
## Part E – Python Analytics (Pandas)

In [1]:
from pathlib import Path
import pandas as pd

cleaned_dir = Path("../data/cleaned") if Path("../data/cleaned").exists() else Path("data/cleaned")

products = pd.read_csv(cleaned_dir / "products_cleaned.csv")
users = pd.read_csv(cleaned_dir / "users_cleaned.csv")
carts = pd.read_csv(cleaned_dir / "carts_cleaned.csv")
cart_items = pd.read_csv(cleaned_dir / "cart_items_cleaned.csv")

# String cleaning/inspection + filtering
cart_items["clean_title"] = cart_items["title"].astype(str).str.strip().str.title()
filtered_items = cart_items[
    (cart_items["discount_percentage"] >= 10)
    & (~filtered_items["clean_title"].str.lower().eq("") if False else ~cart_items["clean_title"].str.lower().eq(""))
]

filtered_items[["cart_id", "clean_title", "quantity", "discount_percentage", "discounted_total"]].head(10)

,cart_id,clean_title,quantity,discount_percentage,discounted_total
0,1,Blue Frock,4,12.13,105.41
1,1,Generic Motorcycle,3,12.10,10547.97
5,2,Apple Iphone Charger,2,18.52,32.58
10,3,Dior J'Adore,2,14.72,153.49
11,3,Party Glasses,5,11.22,88.74
13,4,Dior J'Adore,4,14.72,306.97
14,5,Samsung Galaxy Tab White,4,18.20,1145.17
15,5,Soft Drinks,4,17.48,6.57
17,6,Rolex Datejust Women,5,15.94,46232.96
18,6,Realme Xt,1,11.51,309.71


In [2]:
from pathlib import Path
import pandas as pd

cleaned_dir = Path("../data/cleaned") if Path("../data/cleaned").exists() else Path("data/cleaned")

products = pd.read_csv(cleaned_dir / "products_cleaned.csv")
users = pd.read_csv(cleaned_dir / "users_cleaned.csv")
carts = pd.read_csv(cleaned_dir / "carts_cleaned.csv")
cart_items = pd.read_csv(cleaned_dir / "cart_items_cleaned.csv")

cart_items["clean_title"] = cart_items["title"].astype(str).str.strip().str.title()
filtered_items = cart_items[
    (cart_items["discount_percentage"] >= 10)
    & (cart_items["clean_title"].str.len() > 0)
]

filtered_items[["cart_id", "clean_title", "quantity", "discount_percentage", "discounted_total"]].head(10)

,cart_id,clean_title,quantity,discount_percentage,discounted_total
0,1,Blue Frock,4,12.13,105.41
1,1,Generic Motorcycle,3,12.10,10547.97
5,2,Apple Iphone Charger,2,18.52,32.58
10,3,Dior J'Adore,2,14.72,153.49
11,3,Party Glasses,5,11.22,88.74
13,4,Dior J'Adore,4,14.72,306.97
14,5,Samsung Galaxy Tab White,4,18.20,1145.17
15,5,Soft Drinks,4,17.48,6.57
17,6,Rolex Datejust Women,5,15.94,46232.96
18,6,Realme Xt,1,11.51,309.71


In [3]:
# merging multiple tables 
merged_df = (
    cart_items
    .merge(products, left_on="product_id", right_on="id", how="left", suffixes=("_item", "_prod"))
    .merge(carts, left_on="cart_id", right_on="id", how="left", suffixes=("", "_cart"))
    .merge(users, left_on="user_id", right_on="id", how="left", suffixes=("", "_user"))
)

merged_df[["cart_id", "full_name", "country", "clean_title", "category", "quantity", "discounted_total"]].head(10)

,cart_id,full_name,country,clean_title,category,quantity,discounted_total
0,1,Emily Johnson,United States,Blue Frock,NaN,4,105.41
1,1,Emily Johnson,United States,Generic Motorcycle,NaN,3,10547.97
2,1,Emily Johnson,United States,Iphone 6,NaN,3,839.76
3,1,Emily Johnson,United States,Baseball Ball,NaN,2,17.67
4,2,Michael Williams,United States,Man Short Sleeve Shirt,mens shirts,5,93.12
5,2,Michael Williams,United States,Apple Iphone Charger,NaN,2,32.58
6,3,Sophia Brown,United States,Fish Steak,groceries,1,14.36
7,3,Sophia Brown,United States,Iphone 13 Pro,NaN,1,996.92
8,3,Sophia Brown,United States,Realme X,NaN,1,279.14
9,3,Sophia Brown,United States,Man Short Sleeve Shirt,mens shirts,5,93.12


In [4]:
# lambda, apply and element wise mapping
merged_df["discount_tier"] = merged_df["discount_percentage_item"].apply(
    lambda d: "High (15%+)" if pd.notna(d) and d >= 15 else ("Moderate (5-14%)" if pd.notna(d) and d >= 5 else "Low (<5%)")
)

merged_df["effective_unit_price"] = merged_df.apply(
    lambda r: round(r["discounted_total"] / r["quantity"], 2) if pd.notna(r["quantity"]) and r["quantity"] > 0 else 0.0,
    axis=1,
)

# Element-wise mapping on a numeric slice (modern Pandas replacement for DataFrame.applymap)
price_preview = products[["price", "discounted_price", "stock_value"]].head(5).map(
    lambda val: round(float(val), 2) if pd.notna(val) else val
)

price_preview

,price,discounted_price,stock_value
0,9.99,8.94,885.06
1,19.99,16.35,555.90
2,14.99,13.51,1202.39
3,12.99,11.41,1038.31
4,8.99,7.96,628.84


In [6]:
#groupby 
category_perf = (
    merged_df.groupby("category", dropna=False)
    .agg(
        line_items=("product_id", "count"),
        units_sold=("quantity", "sum"),
        gross_revenue=("total", "sum"),
        net_revenue=("discounted_total", "sum"),
        avg_discount_pct=("discount_percentage_item", "mean"),
    )
    .reset_index()
    .sort_values(by="net_revenue", ascending=False)
)

category_perf

,category,line_items,units_sold,gross_revenue,net_revenue,avg_discount_pct
11,NaN,182,543,1788278.57,1600336.46,12.046374
9,mens watches,13,31,324589.69,297009.12,8.826154
6,laptops,7,24,39599.76,36287.53,8.748571
2,furniture,10,39,26799.61,24381.05,9.708000
8,mens shoes,12,39,4339.61,4153.66,4.428333
1,fragrances,11,29,2429.71,2185.39,8.867273
5,kitchen accessories,49,148,2296.52,2055.74,10.325102
10,mobile accessories,5,14,1639.86,1405.85,14.152000
4,home decoration,11,36,1184.64,1070.53,9.758182
3,groceries,58,172,975.48,890.30,9.435517


In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

cwd = Path.cwd()
root = cwd if (cwd / "data").exists() else cwd.parent
cleaned_dir = root / "data" / "cleaned"

cart_items = pd.read_csv(cleaned_dir / "cart_items_cleaned.csv")
carts = pd.read_csv(cleaned_dir / "carts_cleaned.csv")
products = pd.read_csv(cleaned_dir / "products_cleaned.csv")
users = pd.read_csv(cleaned_dir / "users_cleaned.csv")

df = (
    cart_items
    .merge(carts, left_on="cart_id", right_on="id", how="left", suffixes=("", "_cart"))
    .merge(products, left_on="product_id", right_on="id", how="left", suffixes=("", "_product"))
    .merge(users, left_on="user_id", right_on="id", how="left", suffixes=("", "_user"))
)

if "birth_date" in df.columns:
    df["birth_date"] = pd.to_datetime(df["birth_date"], errors="coerce")
    df["birth_year"] = df["birth_date"].dt.year
    age_years = (pd.Timestamp.today() - df["birth_date"]).dt.days / 365.25
    df["age_group"] = np.select(
        [age_years < 25, age_years < 35, age_years < 50],
        ["Under 25", "25-34", "35-49"],
        default="50+",
    )

if "price" in df.columns:
    df["price_tier"] = np.where(
        df["price"] >= 200,
        "High Price",
        np.where(df["price"] >= 50, "Mid Price", "Low Price"),
    )

out_file = cleaned_dir / "powerbi_dashboard_data.csv"
df.to_csv(out_file, index=False)
print("Exported:", out_file, "| Rows:", len(df), "| Cols:", len(df.columns))


Exported: /Users/macbookair/FinalProject/data/cleaned/powerbi_dashboard_data.csv | Rows: 376 | Cols: 59
